# Finding Places with OpenStreetMap

**Series:** Geocoding (notebook 2 of 3)
**Time:** about 25 minutes

**In this notebook you will:**
- Learn what OpenStreetMap (OSM) tags are
- Download every supermarket in a city with one line of code
- Map the results and turn them into a clean table
- Turn coordinates back into an address (reverse geocoding)

**You should already know:** notebook 01a (GeoDataFrames and `.explore()`).

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## Why OpenStreetMap?

Census data tells you **who** lives in an area. OpenStreetMap tells you **what is there**:
grocery stores, schools, clinics, parks, bus stops. Combining the two is the basis of most
access studies, such as food deserts or distance to health care.

OSM is a free map built by volunteers, so coverage is good in most U.S. cities but not perfect.

## Step 1: Install and import

In [ ]:
!pip install -q osmnx

import osmnx as ox
import pandas as pd

## Step 2: Understand OSM tags

Every feature in OSM is described by **tags**, written as `key=value`:

| What you want | Tag |
|---|---|
| Supermarkets | `shop=supermarket` |
| Schools | `amenity=school` |
| Hospitals | `amenity=hospital` |
| Parks | `leisure=park` |

The full list is on the [OSM Map Features page](https://wiki.openstreetmap.org/wiki/Map_features).
Use tags, not brand names. Searching for "Walmart" will miss stores that are tagged differently.

## Step 3: Download supermarkets

`features_from_place` looks up the place name, then downloads every feature with the tag you ask for.

In [ ]:
place = "West Lafayette, Indiana, USA"
tags = {"shop": "supermarket"}

stores = ox.features_from_place(place, tags=tags)
print(len(stores), "supermarkets found")

## Step 4: Keep only the useful columns

OSM returns many columns. We keep the name and the geometry. Some stores are drawn as building
outlines (polygons), so we convert every store to its center point.

In [ ]:
stores = stores.reset_index()[["name", "geometry"]]
stores["geometry"] = stores.geometry.representative_point()   # one point inside each shape
stores.head()

## Step 5: Map them

In [ ]:
stores.explore(tooltip="name", marker_kwds={"radius": 7}, color="green", tiles="Esri.WorldGrayCanvas")

## Step 6: Make a simple table with coordinates

A plain table of names and coordinates is often all you need for the next step of an analysis.

In [ ]:
stores["Latitude"] = stores.geometry.y
stores["Longitude"] = stores.geometry.x
stores_table = stores[["name", "Latitude", "Longitude"]]
stores_table

## Step 7: Reverse geocoding (coordinates to address)

Sometimes you have GPS points and want a readable address. `Nominatim` is OSM's free geocoder.
Give it a `user_agent` (any short name for your project) and do not send more than one request per second.

In [ ]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="spatialturn_tutorial")
location = geolocator.reverse((40.4237, -86.9212), timeout=30)
print(location.address)

If you get a timeout error, the free server is busy. Wait a minute and run the cell again.

### Under the hood (optional)

`features_from_place` builds a query for the **Overpass API**, the service that searches OSM data.
If you are curious, the same request written by hand looks like this:

```
[out:json];
area[name="West Lafayette"]->.a;
nwr["shop"="supermarket"](area.a);
out center;
```

## Try it yourself

Download all schools (`amenity=school`) in West Lafayette and map them in blue. How many did you find?

<details>
<summary><b>Show a solution</b></summary>

```python
schools = ox.features_from_place(place, tags={"amenity": "school"})
schools = schools.reset_index()[["name", "geometry"]]
schools["geometry"] = schools.geometry.representative_point()
print(len(schools), "schools")
schools.explore(tooltip="name", color="blue", tiles="Esri.WorldGrayCanvas")
```

</details>

In [ ]:
# Write your answer here

## Recap

- OSM features are described by `key=value` tags such as `shop=supermarket`.
- `ox.features_from_place` downloads every feature with a tag inside a place.
- Convert polygons to points with `representative_point()` when you only need locations.
- Reverse geocoding turns coordinates back into an address.

**Next:** [Batch Geocoding and Ambiguous Place Names](https://colab.research.google.com/github/SpatialTurn/DataCollection/blob/main/notebooks/02c_Batch_Geocoding_and_Ambiguous_Names.ipynb)